In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cpu
PyTorch: 2.10.0+cpu


In [2]:
# load encoded data
X = np.load('../data/X_encoded.npy')
y = np.load('../data/y_labels.npy')

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Positive: {y.sum()} ({y.mean()*100:.1f}%)")

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test)

print(f"\nTrain: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

# convert to tensors
X_train_t = torch.FloatTensor(X_train)
X_val_t   = torch.FloatTensor(X_val)
X_test_t  = torch.FloatTensor(X_test)
y_train_t = torch.FloatTensor(y_train)
y_val_t   = torch.FloatTensor(y_val)
y_test_t  = torch.FloatTensor(y_test)

# dataloaders
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32, shuffle=True)
val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=32, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

X shape: (378, 20, 9)
y shape: (378,)
Positive: 187 (49.5%)

Train: 302
Val:   38
Test:  38

Train batches: 10
Val batches:   2


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=20):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class CRISPRTransformer(nn.Module):
    def __init__(self, input_dim=9, d_model=64,
                 nhead=8, num_layers=4, dropout=0.1):
        super().__init__()

        # project input features to d_model
        self.input_proj = nn.Linear(input_dim, d_model)

        # positional encoding
        self.pos_enc = PositionalEncoding(d_model)

        # CLS token — aggregates sequence info for classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers)

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x, return_attention=False):
        batch_size = x.size(0)

        # project to d_model
        x = self.input_proj(x)

        # add positional encoding
        x = self.pos_enc(x)

        # prepend CLS token
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (batch, 21, d_model)

        # transformer
        x = self.transformer(x)

        # use CLS token output for classification
        cls_out = x[:, 0, :]
        return self.classifier(cls_out).squeeze(1)

# create model
model = CRISPRTransformer().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"CRISPR Transformer ready")
print(f"Parameters: {total:,}")
print(f"\nArchitecture:")
print(model)

CRISPR Transformer ready
Parameters: 202,753

Architecture:
CRISPRTransformer(
  (input_proj): Linear(in_features=9, out_features=64, bias=True)
  (pos_enc): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropo

In [4]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=10, factor=0.5)

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            out = model(X_batch)
            loss = criterion(out, y_batch)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            probs = torch.sigmoid(out).cpu().detach().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_batch.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels,
                         (np.array(all_probs) > 0.5).astype(int))
    auc = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, auc

# training
best_auc = 0
history = []
epochs = 100

print(f"Training CRISPR Transformer...")
print(f"\n{'Ep':>4} {'TLoss':>7} {'TAcc':>6} {'TAUC':>6} "
      f"{'VLoss':>7} {'VAcc':>6} {'VAUC':>6}")
print("-" * 48)

for ep in range(epochs):
    tl, ta, tauc = run_epoch(model, train_loader,
                              criterion, optimizer)
    vl, va, vauc = run_epoch(model, val_loader, criterion)
    scheduler.step(vauc)
    history.append([tl, ta, tauc, vl, va, vauc])

    flag = ''
    if vauc > best_auc:
        best_auc = vauc
        torch.save(model.state_dict(),
                   '../data/crispr_model.pth')
        flag = ' ✓'

    if (ep+1) % 10 == 0:
        print(f"{ep+1:>4} {tl:>7.4f} {ta:>6.3f} {tauc:>6.4f} "
              f"{vl:>7.4f} {va:>6.3f} {vauc:>6.4f}{flag}")

print(f"\nBest validation AUC: {best_auc:.4f}")
print(f"Model saved to data/crispr_model.pth")

Training CRISPR Transformer...

  Ep   TLoss   TAcc   TAUC   VLoss   VAcc   VAUC
------------------------------------------------
  10  0.6942  0.500 0.5156  0.6931  0.500 0.7064
  20  0.9814  0.526 0.5458  0.6639  0.500 1.0000
  30  0.6998  0.497 0.5029  0.6941  0.500 0.4889
  40  0.6956  0.477 0.4974  0.6935  0.500 0.3296
  50  0.6899  0.546 0.5622  0.6932  0.500 0.3546
  60  0.6944  0.487 0.4789  0.6933  0.500 0.3393
  70  0.6942  0.503 0.4830  0.6933  0.500 0.3449
  80  0.6914  0.523 0.5363  0.6933  0.500 0.3573
  90  0.6928  0.526 0.5024  0.6932  0.500 0.3573
 100  0.6942  0.510 0.5109  0.6933  0.500 0.3560

Best validation AUC: 1.0000
Model saved to data/crispr_model.pth


In [5]:
from sklearn.model_selection import StratifiedKFold

# simpler model for small dataset
class CRISPRTransformerSmall(nn.Module):
    def __init__(self, input_dim=9, d_model=32,
                 nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=128, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.transformer(x)
        return self.classifier(x[:, 0, :]).squeeze(1)

# 5-fold cross validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []

print("5-fold cross validation...")
print(f"\n{'Fold':>4} {'Best AUC':>10} {'Final Acc':>10}")
print("-" * 28)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr = torch.FloatTensor(X[train_idx])
    y_tr = torch.FloatTensor(y[train_idx])
    X_vl = torch.FloatTensor(X[val_idx])
    y_vl = torch.FloatTensor(y[val_idx])

    tr_loader = DataLoader(TensorDataset(X_tr, y_tr),
                           batch_size=32, shuffle=True)
    vl_loader = DataLoader(TensorDataset(X_vl, y_vl),
                           batch_size=32, shuffle=False)

    fold_model = CRISPRTransformerSmall().to(device)
    fold_opt = optim.Adam(fold_model.parameters(), lr=0.0005,
                          weight_decay=1e-4)
    fold_crit = nn.BCEWithLogitsLoss()

    best_fold_auc = 0
    for ep in range(150):
        run_epoch(fold_model, tr_loader, fold_crit, fold_opt)
        _, va, vauc = run_epoch(fold_model, vl_loader, fold_crit)
        if vauc > best_fold_auc:
            best_fold_auc = vauc

    fold_aucs.append(best_fold_auc)
    print(f"{fold+1:>4} {best_fold_auc:>10.4f} {va:>10.3f}")

print(f"\nMean AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

5-fold cross validation...

Fold   Best AUC  Final Acc
----------------------------
   1     0.9893      0.974
   2     0.9903      0.974
   3     1.0000      0.974
   4     0.9787      0.947
   5     1.0000      0.987

Mean AUC: 0.9916 ± 0.0079


In [6]:
# train final model on full dataset
print("Training final model on full dataset...")

X_full_train, X_final_test, y_full_train, y_final_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

X_ft = torch.FloatTensor(X_full_train)
y_ft = torch.FloatTensor(y_full_train)
full_loader = DataLoader(TensorDataset(X_ft, y_ft),
                         batch_size=32, shuffle=True)

final_model = CRISPRTransformerSmall().to(device)
final_opt = optim.Adam(final_model.parameters(),
                       lr=0.0005, weight_decay=1e-4)
final_crit = nn.BCEWithLogitsLoss()

best_loss = float('inf')
for ep in range(150):
    tl, ta, tauc = run_epoch(final_model, full_loader,
                              final_crit, final_opt)
    if tl < best_loss:
        best_loss = tl
        torch.save(final_model.state_dict(),
                   '../data/crispr_final_model.pth')

# evaluate on held-out test set
X_test_t = torch.FloatTensor(X_final_test)
y_test_t = torch.FloatTensor(y_final_test)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t),
                         batch_size=32, shuffle=False)

final_model.load_state_dict(torch.load('../data/crispr_final_model.pth',
                                        map_location=device))
_, test_acc, test_auc = run_epoch(final_model, test_loader, final_crit)

print(f"\nFinal test results:")
print(f"  AUC:      {test_auc:.4f}")
print(f"  Accuracy: {test_acc*100:.1f}%")
print(f"\n5-fold CV: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"Model saved to data/crispr_final_model.pth")

Training final model on full dataset...

Final test results:
  AUC:      0.9711
  Accuracy: 96.5%

5-fold CV: 0.9916 ± 0.0079
Model saved to data/crispr_final_model.pth
